# 02 - Feature Engineering

Build the matchup-level feature matrix and inspect feature distributions.

In [ ]:

import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from data_loader import build_matchup_dataset, build_team_stats
from feature_engineering import prepare_training_data, impute_features, run_rfe


In [ ]:

print("Building team stats (merging all data sources)...")
team_stats = build_team_stats()
print(f"Team stats shape: {team_stats.shape}")
print(f"Columns: {list(team_stats.columns[:20])}...")


In [ ]:

print("Building matchup dataset...")
matchup_df = build_matchup_dataset(team_stats)
print(f"Matchup dataset shape: {matchup_df.shape}")
print(f"Years available: {sorted(matchup_df['YEAR'].unique())}")
print(f"\nTarget distribution:")
print(matchup_df['TEAM_A_WIN'].value_counts())


In [ ]:

# build feature matrix
X, y = prepare_training_data(matchup_df)
X = impute_features(X)
print(f"Feature matrix: {X.shape}")
print(f"\nFeature list ({len(X.columns)} total):")
for i, c in enumerate(X.columns):
    print(f"  {i+1:3d}. {c}")


In [ ]:

# distribution of the primary predictor: AdjEM delta
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

col = 'delta_badj_em'
if col in X.columns:
    wins = X[col][y == 1]
    losses = X[col][y == 0]
    axes[0].hist(wins, bins=40, alpha=0.6, label='Winner (Team A)', color='steelblue', edgecolor='none')
    axes[0].hist(losses, bins=40, alpha=0.6, label='Loser (Team A)', color='salmon', edgecolor='none')
    axes[0].axvline(0, color='black', linestyle='--')
    axes[0].set_xlabel('Barttorvik AdjEM Delta (A - B)')
    axes[0].set_title('AdjEM Delta: Winner vs Loser')
    axes[0].legend()

axes[1].scatter(X['delta_badj_em'], X['seed_diff'], c=y, cmap='coolwarm', alpha=0.2, s=5)
axes[1].set_xlabel('AdjEM Delta')
axes[1].set_ylabel('Seed Difference (A - B)')
axes[1].set_title('AdjEM Delta vs Seed Difference')

plt.tight_layout()
plt.show()


In [ ]:

# check missing values
missing = X.isna().sum().sort_values(ascending=False)
print("Missing values per feature (top 20):")
print(missing.head(20))


In [ ]:

# run RFE to find optimal feature subset (on training years only)
years = matchup_df['YEAR'].reset_index(drop=True)
train_mask = years <= 2023

print("Running RFECV with 5-fold CV on training years (2008-2023)...")
selected_features, rfecv = run_rfe(X[train_mask], y[train_mask])
print(f"\nOptimal feature count: {len(selected_features)}")
print("Selected features:")
for f in selected_features:
    print(f"  {f}")


In [ ]:

# feature importance from RFECV estimator
if hasattr(rfecv, 'estimator_') and hasattr(rfecv.estimator_, 'coef_'):
    coefs = rfecv.estimator_.coef_[0]
    feat_imp = pd.Series(abs(coefs), index=selected_features).sort_values(ascending=False)
    plt.figure(figsize=(10, 6))
    feat_imp.head(20).plot(kind='barh')
    plt.xlabel('|Coefficient| (Logistic Regression)')
    plt.title('Top 20 Features by Coefficient Magnitude (RFECV)')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
